# 00 — Preparar el dataset

Arma el dataset de entrenamiento con tres fuentes de licencia abierta
(ver `ml/DATASET.md`): `Shopify/product-catalogue` (Apache-2.0, la
principal), Fashion Product Images (MIT) y Amazon Berkeley Objects
(CC BY 4.0) para completar las clases que Shopify casi no tiene. Mapea
todo a las 21 categorías de NovaStore, traduce a español lo que haga
falta y guarda `train.csv`/`val.csv`/`test.csv` en tu Google Drive —
**todos los demás notebooks leen esa misma partición**, así que corre este
primero.

Tiempo estimado en Colab con GPU T4: 15-30 minutos (descarga ~10 GB de
Shopify, pero solo guarda las ~6 500 imágenes elegidas).

In [ ]:
# Si ya corriste esta celda en esta sesión de Colab, puedes saltarla.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_ML = "/content/drive/MyDrive/NovaStore_IA/ml"
    get_ipython().system(f"pip install -q -r {CARPETA_ML}/requirements.txt")
else:
    # Ejecución local (verificación, no Colab): la carpeta ml/ del repo.
    CARPETA_ML = str(Path.cwd())

os.chdir(CARPETA_ML)
sys.path.insert(0, CARPETA_ML)
print("CARPETA_ML =", CARPETA_ML)


In [ ]:
MODO_RAPIDO = False  # True = solo Shopify, 3000 filas revisadas, 5 ejemplos/clase: para probar que corre
CARPETA_DATOS = Path(CARPETA_ML) / "datos"
MAX_POR_CLASE = 5 if MODO_RAPIDO else 500
LIMITE_FILAS = 3000 if MODO_RAPIDO else None

print(f"MODO_RAPIDO={MODO_RAPIDO} · max_por_clase={MAX_POR_CLASE} · limite_filas={LIMITE_FILAS}")


## 1. Elegir productos y bajar sus imágenes

`construir_dataset` lee primero solo el texto y la categoría de cada
fuente, elige hasta `MAX_POR_CLASE` productos por categoría (Shopify
primero, máximo 300 por fuente complementaria) y recién ahí baja las
imágenes de los elegidos. Al final imprime una tabla de ejemplos por
categoría y por fuente.

In [ ]:
from preparar_dataset import construir_dataset

df = construir_dataset(
    max_por_clase=MAX_POR_CLASE,
    limite_filas=LIMITE_FILAS,
    carpeta_salida=CARPETA_DATOS,
)
df.head()


## 2. Traducir a español

Lo que `langdetect` no identifica como español se traduce con
`Helsinki-NLP/opus-mt-en-es` (con caché en `datos/cache/`). Ver la
limitación metodológica (textos en un tercer idioma) en `ml/DATASET.md`.

In [ ]:
from preparar_dataset import traducir_a_espanol

df = traducir_a_espanol(df, carpeta_cache=CARPETA_DATOS / "cache")
df[["titulo", "texto_es"]].sample(min(5, len(df)))


## 3. Partir en train/val/test (70/15/15, semilla 42) y guardar

In [ ]:
from preparar_dataset import guardar_particion, particionar

train, val, test = particionar(df, estratificar=not MODO_RAPIDO)
guardar_particion(train, val, test, CARPETA_DATOS)


## 4. Revisar: algunos ejemplos con imagen

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

muestra = train.sample(min(8, len(train)), random_state=42)
fig, ejes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, fila) in zip(ejes.flat, muestra.iterrows()):
    try:
        ax.imshow(Image.open(fila["ruta_imagen"]))
    except Exception:
        pass
    ax.set_title(f'{fila["categoria"]} ({fila["fuente"]})', fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
